In [0]:
# =============================================================================
# Streaming Table: raw_orders (Streaming Query)
# =============================================================================
from pyspark.pipelines import materialized_view, table, create_streaming_table, append_flow
from pyspark.sql import functions as F
create_streaming_table("raw_orders", comment="raw orders table")

@append_flow(
    target="raw_orders",
    comment="Raw orders streaming from S3 via Autoloader"
)
def raw_orders():
    """
    Streaming table for raw order data using Autoloader.

    Source: S3 bucket with JSON files
    Location: s3://dsas-datasets/orders/
    Pattern: Autoloader (cloudFiles) for automatic schema inference and incremental loading
    Runs: Continuously

    Schema:
        order_id: string (PK)
        customer_id: int (FK to customers)
        product_id: string
        quantity: int
        amount: decimal(10,2)
        order_date: timestamp

    Data Quality:
        - Filter quantity > 0 (no invalid orders)
        - Explicit schema casting for type safety

    Note: Autoloader automatically handles:
        - Schema inference and evolution
        - Exactly-once processing semantics
        - Incremental file discovery
        - Checkpoint management
    """
    return (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.schemaLocation",
                    "/Volumes/main/default/checkpoints/raw_orders_schema")
            .option("cloudFiles.schemaHints",
                    "order_id STRING, customer_id INT, product_id STRING, quantity INT, amount DECIMAL(10,2), order_date TIMESTAMP")
            .load("s3://dsas-datasets/orders/")
            .select(
                F.col("order_id").cast("string"),
                F.col("customer_id").cast("int"),
                F.col("product_id").cast("string"),
                F.col("quantity").cast("int"),
                F.col("amount").cast("decimal(10,2)"),
                F.col("order_date").cast("timestamp")
            )
            .filter(F.col("quantity") > 0))